## Part 1: Handcrafted Features + SVM Classifier

Цель эксперимента:
Воспроизведение базовой части статьи, где используются ручные (handcrafted) акустические признаки ComParE и eGeMAPS, извлекаемые с помощью OpenSMILE, и классификатор SVM.

Ссылка на статью:
"Automatic Parkinson’s disease detection from speech: Layer selection vs adaptation of foundation models"
### === Контекст ===
OpenSMILE — это инструмент для извлечения акустических признаков из аудио. Он предоставляет несколько готовых конфигураций для извлечения признаков:

<details>
    <summary>1. eGeMAPS (Extended Geneva Minimalistic Acoustic Parameter Set):</summary>

   * Это компактный набор из 88 признаков, включающий параметры, связанные с основным тоном (F0), энергией, спектральными характеристиками, голосовым возбуждением (voicing), дрожанием (jitter), дрожанием амплитуды (shimmer) и другими.
   * Признаки считаются "минималистичными", так как разработаны для универсального использования в задачах анализа эмоций, патологий речи и т.д.
   * В статье: EGEMAPS LLD×F — 88 статических признаков, полученных как функционалы (статистики) над 23 LLD (Low-Level Descriptors).

</details>

<details>
    <summary>2. ComParE (Computational Paralinguistics Challenge):</summary>

   * Это более обширный набор из 6373 признаков, включающий большое количество LLD, таких как MFCC, спектральные признаки, F0, шумность, HNR и т.д.
   * Функционалы (среднее, стандартное отклонение, квантили и др.) вычисляются над временными контурами этих признаков.
   * В статье: COMPARE LLD×F — 6373 признака на уровень высказывания (turn-level).

</details>

Оба признака извлекаются на уровне высказывания (utterance/turn level) путем вычисления статистических функционалов (например, mean, std, min, max, kurtosis, percentiles) над временными контурами LLD.
### === План эксперимента ===
1. Загрузка аудиофайлов (PC-GITA датасет — заглушка)
2. Извлечение признаков eGeMAPS и ComParE с помощью OpenSMILE
3. Обучение SVM с кросс-валидацией для настройки гиперпараметров
4. Оценка по метрикам: accuracy, F1, sensitivity, specificity
### === Гиперпараметры из статьи ===
- Классификатор: SVM
- Настройка: grid search по C, gamma, kernel (на валидационной выборке)
- Оценка: стратифицированная 10-фолдовая speaker-independent кросс-валидация
- Метрики: mean и std по 10 фолдам

In [8]:
import os
import numpy as np
import pandas as pd
import sklearn
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, recall_score
from sklearn.preprocessing import StandardScaler
from imblearn.metrics import specificity_score
import opensmile

In [ ]:
# === Настройка путей ===
dataset_root = "path/to/PC-GITA"  # ЗАГЛУШКА — замените на реальный путь после загрузки
data_list = []  # Список: путь к аудио, метка (0=HC, 1=PD)

# === Функция извлечения признаков ===
def extract_features(audio_path, feature_set="comparE"):
    """
    Извлекает признаки ComParE или eGeMAPS из аудиофайла.
    """
    if feature_set == "comparE":
        smile = opensmile.Smile(
            feature_set=opensmile.FeatureSet.ComParE_2016,
            feature_level=opensmile.FeatureLevel.Functionals,
        )
    elif feature_set == "eGeMAPS":
        smile = opensmile.Smile(
            feature_set=opensmile.FeatureSet.eGeMAPSv02,
            feature_level=opensmile.FeatureLevel.Functionals,
        )
    else:
        raise ValueError("Unknown feature set")

    features = smile.process_file(audio_path)
    return features.values.flatten()  # Вектор признаков


# === Собираем данные (заглушка) ===
# Предполагается, что у вас есть структура:
# PC-GITA/
#   PD/
#     pd_001.wav, ...
#   HC/
#     hc_001.wav, ...

for label, group in enumerate(['HC', 'PD']):
    group_path = os.path.join(dataset_root, group)
    if not os.path.exists(group_path):
        print(f"[WARNING] Directory not found: {group_path}")
        continue
    for file in os.listdir(group_path):
        if file.endswith('.wav'):
            data_list.append((os.path.join(group_path, file), label))

if len(data_list) == 0:
    raise FileNotFoundError("No audio files found. Check dataset path and structure.")

df_data = pd.DataFrame(data_list, columns=['audio_path', 'label'])

# === Извлечение признаков (пример для eGeMAPS) ===
print("Extracting eGeMAPS features...")
X = []
y = []
for idx, row in df_data.iterrows():
    try:
        feats = extract_features(row['audio_path'], feature_set="eGeMAPS")
        X.append(feats)
        y.append(row['label'])
    except Exception as e:
        print(f"Error processing {row['audio_path']}: {e}")
        continue

X = np.array(X)
y = np.array(y)

# === Нормализация ===
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# === Кросс-валидация и подбор гиперпараметров ===
print("Performing GridSearchCV on SVM...")
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=1337)

param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1],
    'kernel': ['rbf', 'linear']
}

svm = SVC(random_state=1337)
grid_search = GridSearchCV(svm, param_grid, cv=skf, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_scaled, y)

best_svm = grid_search.best_estimator_
print(f"Best params: {grid_search.best_params_}")

# === Оценка по фолдам ===
accuracies = []
f1_scores = []
sensitivities = []
specificities = []

for train_idx, test_idx in skf.split(X_scaled, y):
    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Обучение на лучших параметрах
    clf = SVC(**grid_search.best_params_, random_state=1337)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    accuracies.append(accuracy_score(y_test, y_pred))
    f1_scores.append(f1_score(y_test, y_pred))
    sensitivities.append(recall_score(y_test, y_pred, pos_label=1))  # PD как positive
    specificities.append(recall_score(y_test, y_pred, pos_label=0))  # HC как negative

# === Вывод результатов ===
print("\n=== Results (eGeMAPS + SVM) ===")
print(f"Accuracy: {np.mean(accuracies):.2f} (±{np.std(accuracies):.2f})")
print(f"F1-score: {np.mean(f1_scores):.2f} (±{np.std(f1_scores):.2f})")
print(f"Sensitivity (PD): {np.mean(sensitivities):.2f} (±{np.std(sensitivities):.2f})")
print(f"Specificity (HC): {np.mean(specificities):.2f} (±{np.std(specificities):.2f})")

# Вы можете повторить для ComParE, заменив feature_set="comparE"


### Part 2: Speech Foundation Models (SFM) + LoRA fine-tuning

Цель эксперимента:
Воспроизведение второй части статьи, где используются крупные предобученные речевые модели (SFM): Wav2Vec2.0, XLS-R, Whisper-small, с тремя подходами:
1. Layer selection (замороженная модель + классификатор поверх одного слоя)
2. Full fine-tuning (обновление всех параметров)
3. LoRA (Low-Rank Adaptation) — параметро-эффективное дообучение

### === Гиперпараметры из статьи ===
- Модели: Wav2Vec2.0-base, XLS-R-300M, Whisper-small
- Оптимизатор: Adam
- Learning rate: 1e-4 (для layer probing), 1e-5 (для fine-tuning)
- Batch size: 4, gradient accumulation: 8
- Seed: 1337
- Classifier head: 1 скрытый слой (256 ReLU), выходной слой (2 softmax)
- LoRA rank: {4, 8, 16} — лучший результат при r=16 для Whisper
- CNN-энкодеры заморожены, обучаются только transformer-слои

### === Важные замечания ===
- Whisper: используются только encoder-слои (decoder отбрасывается)
- Среднее пулинг по временным фреймам перед классификатором
- Speaker-independent 10-fold CV

### === Заглушки ===
1. Датасет PC-GITA — недоступен, замените путь
2. SFM модели — загружаются из Hugging Face
3. LoRA — требует библиотеку peft. Если не удается установить — используйте заглушку

In [12]:
import torchaudio
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, AutoModel
from peft import LoraConfig, get_peft_model
import gc

In [ ]:
# === Настройки ===
device = "cuda" if torch.cuda.is_available() else "cpu"
dataset_root = "path/to/PC-GITA"  # ЗАГЛУШКА
seed = 1337
torch.manual_seed(seed)
np.random.seed(seed)

class_names = ['HC', 'PD']
label_to_id = {name: idx for idx, name in enumerate(class_names)}

# === Загрузка данных ===
data_list = []
for label, group in enumerate(['HC', 'PD']):
    group_path = os.path.join(dataset_root, group)
    if not os.path.exists(group_path):
        print(f"[WARNING] Directory not found: {group_path}")
        continue
    for file in os.listdir(group_path):
        if file.endswith('.wav'):
            data_list.append((os.path.join(group_path, file), label))

df_data = pd.DataFrame(data_list, columns=['audio_path', 'label'])

# === Dataset класс ===
class SpeechDataset(Dataset):
    def __init__(self, df, processor, max_duration=30.0, sample_rate=16000):
        self.df = df
        self.processor = processor
        self.max_duration = max_duration
        self.sample_rate = sample_rate

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio_path = row['audio_path']
        label = row['label']

        # Загрузка аудио
        waveform, sr = torchaudio.load(audio_path)
        if sr != self.sample_rate:
            waveform = torchaudio.transforms.Resample(sr, self.sample_rate)(waveform)

        # Усечение или дополение
        max_length = int(self.max_duration * self.sample_rate)
        if waveform.shape[1] > max_length:
            waveform = waveform[:, :max_length]
        else:
            pad_length = max_length - waveform.shape[1]
            waveform = torch.nn.functional.pad(waveform, (0, pad_length))

        # Преобразование в признаки
        inputs = self.processor(waveform.squeeze().numpy(), sampling_rate=self.sample_rate, return_tensors="pt", padding=True)

        return inputs.input_values[0], torch.tensor(label, dtype=torch.long)

# === Класс SFM с классификатором ===
class SFMClassifier(nn.Module):
    def __init__(self, model_name, num_labels=2, freeze_cnn=True):
        super().__init__()
        self.sfm = AutoModel.from_pretrained(model_name)
        self.classifier = nn.Sequential(
            nn.Linear(self.sfm.config.hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_labels)
        )

        # Заморозка CNN-слоев (если есть)
        if freeze_cnn and hasattr(self.sfm, 'feature_extractor'):
            for param in self.sfm.feature_extractor.parameters():
                param.requires_grad = False

    def forward(self, input_values):
        outputs = self.sfm(input_values)
        # Mean pooling по временной оси
        hidden_states = outputs.last_hidden_state
        pooled = torch.mean(hidden_states, dim=1)
        return self.classifier(pooled)

def apply_lora(model, rank=4, alpha=16, lora_dropout=0.1, target_modules=["q_proj", "v_proj"]):
    config = LoraConfig(r=rank, lora_alpha=alpha, target_modules=target_modules, lora_dropout=lora_dropout, bias="none")
    model = get_peft_model(model, config)
    return model

# === Обучение модели ===
def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for batch_idx, (x, y) in enumerate(dataloader):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

def evaluate(model, dataloader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
    return accuracy_score(all_labels, all_preds), f1_score(all_labels, all_labels, average='binary'), recall_score(all_labels, all_preds, pos_label=1), recall_score(all_labels, all_preds, pos_label=0)

# === Кросс-валидация по слоям (для layer selection) ===
def layer_selection(model_name, df_data, processor_name, num_layers):
    """
    Идея: обучить классификатор на выходе каждого слоя и выбрать лучший по валидации.
    В статье: используется кросс-валидация для оценки каждого слоя.
    """
    print(f"\n=== Layer Selection for {model_name} ===")
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=seed)

    layer_scores = {i: [] for i in range(num_layers)}

    for fold, (train_idx, val_idx) in enumerate(skf.split(df_data, df_data['label'])):
        print(f"Fold {fold+1}/10")
        df_train = df_data.iloc[train_idx]
        df_val = df_data.iloc[val_idx]

        processor = AutoProcessor.from_pretrained(processor_name)
        train_dataset = SpeechDataset(df_train, processor)
        val_dataset = SpeechDataset(df_val, processor)
        train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)

        # Обучение классификатора для каждого слоя
        base_model = AutoModel.from_pretrained(model_name).to(device)
        base_model.eval()

        for layer_idx in range(num_layers):
            # Заглушка: в реальности нужно извлекать выходы слоя
            # Это требует модификации модели для возврата внутренних состояний
            print(f"  Layer {layer_idx} - not implemented (requires layer-wise output)")

    print("Layer selection requires custom model to output hidden states per layer.")
    print("Recommended: Use `output_hidden_states=True` and extract layer outputs.\n")

# === Основной эксперимент ===
print("=== SFM + LoRA Experiment Reproduction ===")
print("Note: This is a structural reproduction. Full implementation requires access to models and dataset.")

# === 1. Layer Selection (пример для Wav2Vec2) ===
layer_selection(
    model_name="facebook/wav2vec2-base",
    processor_name="facebook/wav2vec2-base",
    df_data=df_data,
    num_layers=12
)

# === 2. LoRA fine-tuning пример ===
print("=== LoRA Fine-tuning (example setup) ===")
print("Model: openai/whisper-small")

# Загрузка модели
model = SFMClassifier("openai/whisper-small", freeze_cnn=True)  # Только encoder
model = apply_lora(model, rank=16)
model.to(device)

# Оптимизатор и loss
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
criterion = nn.CrossEntropyLoss()

# === Вывод ===
print("3. Для layer selection: модифицируйте модель для вывода скрытых состояний по слоям")
print("5. Воспроизведите 10-фолдовую speaker-independent CV\n")